In [2]:
import pandas as pd
import numpy as np
import json
import glob
import os

### Parse each subset of data

In [35]:
# Load file directories
# part = 'aa'
folder_path = 'mc67'
split_folder = f'../task2/{folder_path}/split'
# response_filepath = '../data/data.info.labelled'
concat_folder = f'../task2/{folder_path}/concat'
os.makedirs(concat_folder, exist_ok=True)

In [36]:
# Helper Functions
def load_json_row(filepath):
    with open(filepath, 'r') as f:
        for line in f:
            if line.strip(): 
                yield json.loads(line)

def extract_first3_from_row(row):
    for transcript_id, v1 in row.items():
        for position, v2 in v1.items():
            for sequence, v3 in v2.items():
                first3 = [sublist[:9] for sublist in v3]
                return transcript_id, position, sequence, first3
    return None, None, None, []

def remove_outliers(data):
    if not data:
        return data
    
    q1 = np.percentile(data, 25)
    q3 = np.percentile(data, 75)

    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    return [x for x in data if lower <= x <= upper]

def aggregate_mean(data):
    return np.mean(data) if data else np.nan

def aggregate_median(x):
    return np.median(x) if x else np.nan


def compute_iqr(x):
    if len(x) == 0:
        return np.nan
    return np.percentile(x, 75) - np.percentile(x, 25)


In [ ]:
# List column names
columns = ['transcript_id', 'transcript_position', 'sequence', 
           'mean_0', 'std_0', 'dwell_0',
           'mean_1', 'std_1', 'dwell_1', 
           'mean_2', 'std_2', 'dwell_2',
           'mean_0_iqr', 'std_0_iqr', 'dwell_0_iqr',
           'mean_1_iqr', 'std_1_iqr', 'dwell_1_iqr',
           'mean_2_iqr', 'std_2_iqr', 'dwell_2_iqr']

count = 0

# Loop through all files in the folder and process them
for filepath in glob.glob(f"{split_folder}/*"):
    part_name = os.path.basename(filepath).replace('.json', '')
    out_csv = f"{concat_folder}/concat_agg_{part_name}.csv"

    # Ensure header is written by initializing CSV with columns only
    pd.DataFrame(columns=columns).to_csv(out_csv, index=False)

    count = 0
    for row in load_json_row(filepath):
        transcript_id, position, sequence, features = extract_first3_from_row(row)
        if not features:
            continue

        feats = []
        for i in range(9):
            col = [x[i] for x in features if len(x) > i]
            feats.append(col)

        # Extract Median and IQR for each feature
        aggs_median = [aggregate_median(f) for f in feats]
        aggs_iqr = [compute_iqr(f) for f in feats]

        row_dict = {
            'transcript_id': transcript_id,
            'transcript_position': position,
            'sequence': sequence,

            # Median
            'mean_0': aggs_median[0],
            'std_0': aggs_median[1],
            'dwell_0': aggs_median[2],
            'mean_1': aggs_median[3],
            'std_1': aggs_median[4],
            'dwell_1': aggs_median[5],
            'mean_2': aggs_median[6],
            'std_2': aggs_median[7],
            'dwell_2': aggs_median[8],

            # IQR
            'mean_0_iqr': aggs_iqr[0],
            'std_0_iqr': aggs_iqr[1],
            'dwell_0_iqr': aggs_iqr[2],
            'mean_1_iqr': aggs_iqr[3],
            'std_1_iqr': aggs_iqr[4],
            'dwell_1_iqr': aggs_iqr[5],
            'mean_2_iqr': aggs_iqr[6],
            'std_2_iqr': aggs_iqr[7],
            'dwell_2_iqr': aggs_iqr[8]
        }

        # Append row to CSV
        pd.DataFrame([row_dict]).to_csv(out_csv, mode='a', header=False, index=False)
        count += 1
        print(f"{count} rows written for {part_name}")

### Merge all the parts together

In [38]:
# Merge all parts together
glob_folder_path = f"../task2/{folder_path}/concat"

# Get all CSV files in the folder
csv_files = glob.glob(f"{glob_folder_path}/*.csv")

# Read and combine them into a single DataFrame
df_list = [pd.read_csv(file) for file in csv_files]
combined_df = pd.concat(df_list, ignore_index=True)

print(f"Loaded {len(csv_files)} files.")

Loaded 10 files.


In [39]:
combined_df

,transcript_id,transcript_position,sequence,mean_0,std_0,dwell_0,mean_1,std_1,dwell_1,mean_2,...,dwell_2,mean_0_iqr,std_0_iqr,dwell_0_iqr,mean_1_iqr,std_1_iqr,dwell_1_iqr,mean_2_iqr,std_2_iqr,dwell_2_iqr
0,ENST00000384629.1,72,ATAACAC,0.005573,1.803895,82.20,0.003670,2.303636,93.80,0.011083,...,89.8,0.000000,0.000000,0.000,0.000000,0.000000,0.000,0.000000,0.000000,0.000
1,ENST00000612463.1,78,AGGACAC,0.003650,2.631000,117.10,0.005259,9.222442,116.30,0.002660,...,82.5,0.000000,0.000000,0.000,0.000000,0.000000,0.000,0.000000,0.000000,0.000
2,ENST00000636484.1,170,AGGACCG,0.021942,8.316000,117.80,0.009684,8.831125,124.70,0.007970,...,84.2,0.021373,2.993000,2.000,0.003765,1.887575,5.300,0.002621,3.926091,3.200
3,ENST00000636484.1,218,AGAACCT,0.009721,11.184000,128.15,0.006305,4.542000,96.60,0.007863,...,86.3,0.004089,3.795770,3.575,0.002820,1.646750,1.775,0.002862,0.772965,1.625
4,ENST00000636484.1,226,CAAACAA,0.010567,2.254230,106.65,0.007970,3.384500,100.30,0.007198,...,86.8,0.006936,1.007060,2.925,0.007137,1.744000,1.675,0.009317,0.570753,1.900
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,ENST00000506254.5,2196,CTAACCA,0.005153,1.576833,90.80,0.005520,2.263000,92.00,0.004320,...,85.8,0.003210,0.508575,1.800,0.002985,0.406852,1.700,0.002825,0.631000,1.750
99996,ENST00000506254.5,2218,TAAACAA,0.005971,2.363905,102.20,0.002831,2.380633,97.15,0.003831,...,88.8,0.001496,0.372345,0.900,0.000511,0.294633,2.450,0.000379,0.443960,1.900
99997,ENST00000506254.5,2223,AAAACTC,0.004134,1.718000,107.30,0.004536,2.820556,100.80,0.009300,...,90.4,0.000830,0.455114,2.750,0.003485,0.568500,3.250,0.002660,0.323500,3.100
99998,ENST00000506254.5,2276,CAAACTT,0.005168,2.375844,106.75,0.006200,3.184538,107.15,0.003085,...,92.2,0.002472,0.255156,0.150,0.002327,0.200462,2.150,0.000283,0.068842,0.800


In [ ]:
# combined_df.to_csv(f'../task2/{folder_path}/{folder_path}_agg.csv', index=False)

### Join with response label

In [41]:
response_df = pd.read_csv('../data/data.info.labelled')
merged_df = pd.merge(combined_df, response_df, on=['transcript_id', 'transcript_position'], how='left')
merged_df

,transcript_id,transcript_position,sequence,mean_0,std_0,dwell_0,mean_1,std_1,dwell_1,mean_2,...,std_0_iqr,dwell_0_iqr,mean_1_iqr,std_1_iqr,dwell_1_iqr,mean_2_iqr,std_2_iqr,dwell_2_iqr,gene_id,label
0,ENST00000384629.1,72,ATAACAC,0.005573,1.803895,82.20,0.003670,2.303636,93.80,0.011083,...,0.000000,0.000,0.000000,0.000000,0.000,0.000000,0.000000,0.000,NaN,NaN
1,ENST00000612463.1,78,AGGACAC,0.003650,2.631000,117.10,0.005259,9.222442,116.30,0.002660,...,0.000000,0.000,0.000000,0.000000,0.000,0.000000,0.000000,0.000,NaN,NaN
2,ENST00000636484.1,170,AGGACCG,0.021942,8.316000,117.80,0.009684,8.831125,124.70,0.007970,...,2.993000,2.000,0.003765,1.887575,5.300,0.002621,3.926091,3.200,NaN,NaN
3,ENST00000636484.1,218,AGAACCT,0.009721,11.184000,128.15,0.006305,4.542000,96.60,0.007863,...,3.795770,3.575,0.002820,1.646750,1.775,0.002862,0.772965,1.625,NaN,NaN
4,ENST00000636484.1,226,CAAACAA,0.010567,2.254230,106.65,0.007970,3.384500,100.30,0.007198,...,1.007060,2.925,0.007137,1.744000,1.675,0.009317,0.570753,1.900,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,ENST00000506254.5,2196,CTAACCA,0.005153,1.576833,90.80,0.005520,2.263000,92.00,0.004320,...,0.508575,1.800,0.002985,0.406852,1.700,0.002825,0.631000,1.750,NaN,NaN
99996,ENST00000506254.5,2218,TAAACAA,0.005971,2.363905,102.20,0.002831,2.380633,97.15,0.003831,...,0.372345,0.900,0.000511,0.294633,2.450,0.000379,0.443960,1.900,NaN,NaN
99997,ENST00000506254.5,2223,AAAACTC,0.004134,1.718000,107.30,0.004536,2.820556,100.80,0.009300,...,0.455114,2.750,0.003485,0.568500,3.250,0.002660,0.323500,3.100,NaN,NaN
99998,ENST00000506254.5,2276,CAAACTT,0.005168,2.375844,106.75,0.006200,3.184538,107.15,0.003085,...,0.255156,0.150,0.002327,0.200462,2.150,0.000283,0.068842,0.800,NaN,NaN


In [42]:
# merged_df.to_csv(f'../data/{folder_path}/test2_agg_mean.csv', index=False)